# 📡 CONECTARSE A LOS DATOS

**Taller de IA · 7° TIC · CTP Antonio Martín Marte**

Su agente necesita recibir datos de algún lado. Acá se conectan a una **fuente de datos
que está en internet**, igual que se conectarían a un sensor real.

Cada vez que lo ejecuten van a recibir **la lectura de este momento**. Si lo corren
dentro de un rato, el valor va a ser otro.

---
⚠️ **Son datos simulados, con fines educativos.** No son mediciones reales de nada.

## B1 — Elegí tu fuente

In [ ]:
MI_FUENTE = "aire"    # aire / agua / uv / ruido

import requests
from datetime import datetime

BASE = "https://mariogustavoiturriz.github.io/profegusADIPES/datos/"

r = requests.get(BASE + MI_FUENTE + ".json")
FUENTE = r.json()

print("Conectado a:", FUENTE["fuente"])
print("Mide:", FUENTE["campo_principal"], "en", FUENTE["unidad"])
print("Lecturas disponibles:", FUENTE["total_lecturas"])

## B2 — Leer lo de ahora

La fuente tiene una lectura cada 5 minutos a lo largo del día. Esta función busca
**la que corresponde a la hora actual**.

In [ ]:
def leer_ahora():
    """Devuelve la lectura que corresponde al momento actual."""
    ahora = datetime.now()
    minutos = ahora.hour * 60 + ahora.minute
    indice = (minutos // 5) % FUENTE["total_lecturas"]
    return FUENTE["lecturas"][indice]


def como_texto(lectura):
    """La lectura en el formato que le vas a pasar al agente."""
    partes = []
    for clave, valor in lectura.items():
        if clave in ("minuto_del_dia",):
            continue
        partes.append(str(clave) + ": " + str(valor))
    return "\n".join(partes)


lectura = leer_ahora()
print(como_texto(lectura))

## B3 — Leer un momento cualquiera

Para probar su agente no conviene esperar a que pase algo interesante.
Con esto pueden pedir **la lectura de cualquier hora del día**.

In [ ]:
def leer_a_las(hora, minuto=0):
    """Devuelve la lectura de una hora determinada. Ej: leer_a_las(17, 30)"""
    indice = ((hora * 60 + minuto) // 5) % FUENTE["total_lecturas"]
    return FUENTE["lecturas"][indice]


# Probá distintos momentos y mirá como cambia
for h in [3, 9, 13, 17, 21]:
    l = leer_a_las(h)
    campo = FUENTE["campo_principal"]
    print(l["hora"], "->", campo, l[campo], "|", l["tendencia"], "| atipico:", l["es_atipico"])

### 🔍 Busquen el momento difícil

En cada fuente hay **momentos tranquilos y momentos donde pasa algo**.
Recorran el día y encuentren dónde se complica: ahí es donde su agente tiene que
funcionar bien.

Un agente que solo acierta cuando todo está normal no sirve.

In [ ]:
campo = FUENTE["campo_principal"]
valores = [(l["hora"], l[campo], l["es_atipico"]) for l in FUENTE["lecturas"]]

# los 8 momentos con el valor mas alto
peores = sorted(valores, key=lambda x: -x[1])[:8]
print("Momentos con el valor mas alto:")
for h, v, a in peores:
    print("  ", h, "->", v, "(atipico:", a + ")")

---
## B4 — Conectarlo con su agente

Esto es lo que van a hacer de verdad: la lectura entra, el agente decide.

Peguen arriba su `consultar_ia`, su `TABLA` y su `INSTRUCCION`.

---
## B4 — La conexión con la IA

La de siempre. Pegá tu clave y ejecutá.

In [ ]:
API_KEY = "pega_aca_tu_api_key"


def consultar_ia(instruccion, mensaje, temperatura=0.2):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {"Authorization": "Bearer " + API_KEY,
               "Content-Type": "application/json"}

    datos = {"model": "openai/gpt-oss-120b",
             "messages": [{"role": "system", "content": instruccion},
                          {"role": "user",   "content": mensaje}],
             "temperature": temperatura}

    r = requests.post(url, headers=headers, json=datos)
    return r.json()["choices"][0]["message"]["content"]


print("Funcion lista.")

## B5 — La tabla y la instrucción del agente

Acá va **su** tabla, la que sacaron de los documentos en la Parte 1.

In [ ]:
TABLA = """
contaminante: monoxido de carbono | limite: 9 ppm  | exposicion: 8 horas | fuente: (completar)
contaminante: monoxido de carbono | limite: 35 ppm | exposicion: 1 hora  | fuente: (completar)
"""

INSTRUCCION = """Sos un agente que interpreta mediciones ambientales.

Esta es tu tabla de referencia:
""" + TABLA + """

Vas a recibir una medicion. Comparala con la tabla y respondé EXACTAMENTE asi,
sin agregar nada mas:

nivel: (normal / atencion / alerta)
accion: (que hay que hacer, en pocas palabras)
mensaje: (explicacion clara para una persona, dos oraciones)
razon: (contra que valor de la tabla lo comparaste)

Si la medicion no alcanza para decidir, poné nivel: sin_datos y explicá que falta.
"""

print("Instruccion lista.")

## B6 — El agente andando

Una lectura entra, el agente decide.

In [ ]:
lectura = leer_ahora()
# para probar un momento dificil:
# lectura = leer_a_las(17, 30)

print("ENTRA:")
print(como_texto(lectura))
print()
print("EL AGENTE DECIDE:")
print(consultar_ia(INSTRUCCION, como_texto(lectura)))

## B7 — El bucle

Como en las clases anteriores: se queda funcionando y va decidiendo cada tanto.

Para probar en clase poné `CADA_CUANTO = 20` y `CUANTAS_VECES = 3`.
No lo dejes en una hora mientras estan mirando.

In [ ]:
import time

CADA_CUANTO = 20      # segundos entre una lectura y otra
CUANTAS_VECES = 3     # cuantas veces revisa


for vuelta in range(CUANTAS_VECES):
    lectura = leer_ahora()
    decision = consultar_ia(INSTRUCCION, como_texto(lectura))

    print("=" * 55)
    print("Vuelta", vuelta + 1, "-", lectura["hora"])
    print(decision)

    if vuelta < CUANTAS_VECES - 1:
        time.sleep(CADA_CUANTO)

print("=" * 55)
print("Listo.")

### Probarlo con los momentos difíciles

Antes de dar el agente por terminado: pasale las horas donde pasa algo
y fijate si decide bien. Un agente que solo acierta de madrugada no sirve.

In [ ]:
for h, m in [(4, 0), (11, 0), (17, 30), (21, 0)]:
    lectura = leer_a_las(h, m)
    decision = consultar_ia(INSTRUCCION, como_texto(lectura))

    print("-" * 55)
    print(lectura["hora"], "->", FUENTE["campo_principal"], lectura[FUENTE["campo_principal"]])
    print(decision)

---
## Para el equipo de feria

Esta fuente es **provisoria**: sirve para trabajar mientras deciden qué datos van a
tomar del ESP32 y cómo se los va a entregar Ciencia de Datos.

Cuando eso esté definido, lo único que cambia es de dónde sale la lectura: reemplazan
`leer_ahora()` por la función que lee su sensor. **El agente no se toca**, porque
recibe el mismo formato.

Eso es justamente lo bueno de haberlo separado así.

---
## Si su tema no está en la lista

Las cuatro fuentes son de medición. Si su proyecto trabaja con **situaciones** en vez
de números —derechos, reclamos, normativa— no necesitan esto: escriban ustedes los
casos de prueba, como hicieron con las trampas de la defensa.

Lo importante es el formato: que el agente reciba siempre lo mismo, venga de donde venga.